# Adversarial Search: Solving Tic-Tac-Toe with Monte Carlo Tree Search

## Introduction 

Multiplayer games can be implemented as:
1. Nondeterministic actions: The opponent is seen as part of an environment with nondeterministic actions. Non-determinism is the result of the unknown opponent's moves. 
2. Optimal Decisions: Minimax search (search complete game tree) and alpha-beta pruning.
3. Heuristic Alpha-Beta Tree Search: Cut off tree search and use heuristic to estimate state value. 
4. __Monte Carlo Search:__ Simulate playouts to estimate state value. 

Here we will implement search for Tic-Tac-Toe (see [rules](https://en.wikipedia.org/wiki/Tic-tac-toe)). The game is a __zero-sum game__: Win by x results in +1, win by o in -1 and a tie has a value of 0. Max plays x and tries to maximize the outcome while Min plays o and tries to minimize the outcome.   

We will implement
* Monte Carlo Tree Search with upper confidence bound (UCB1).

## The board

I represent the board as a vector of length 9. The values are `' ', 'x', 'o'`.  

In [284]:
import numpy as np
import pandas as pd
import math

In [285]:
def empty_board():
    return [' '] * 9

board = empty_board()
display(board)

[' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ']

Some helper functions.

In [286]:
def show_board(board):
    """display the board"""
    b = np.array(board).reshape((3,3))
    print(b)

board = empty_board()
show_board(board)    

print()
print("Add some x's")
board[0] = 'x'; board[3] = 'x'; board[6] = 'x';  
show_board(board)

[[' ' ' ' ' ']
 [' ' ' ' ' ']
 [' ' ' ' ' ']]

Add some x's
[['x' ' ' ' ']
 ['x' ' ' ' ']
 ['x' ' ' ' ']]


In [287]:
def check_win(board):
    """check the board and return one of x, o, d (draw), or n (for next move)"""
    
    board = np.array(board).reshape((3,3))
    
    diagonals = np.array([[board[i][i] for i in range(len(board))], 
                          [board[i][len(board)-i-1] for i in range(len(board))]])
    
    for a_board in [board, np.transpose(board), diagonals]:
        for row in a_board:
            if len(set(row)) == 1 and row[0] != ' ':
                return row[0]
    
    # check for draw
    if(np.sum(board == ' ') < 1):
        return 'd'
    
    return 'n'

show_board(board)
print('Win? ' + check_win(board))

print()
show_board(empty_board())
print('Win? ' + check_win(empty_board()))

[['x' ' ' ' ']
 ['x' ' ' ' ']
 ['x' ' ' ' ']]
Win? x

[[' ' ' ' ' ']
 [' ' ' ' ' ']
 [' ' ' ' ' ']]
Win? n


In [288]:
def get_actions(board):
    """return possible actions as a vector ot indices"""
    return np.where(np.array(board) == ' ')[0].tolist()

    # randomize the action order
    #actions = np.where(np.array(board) == ' ')[0]
    #np.random.shuffle(actions)
    #return actions.tolist()


show_board(board)
get_actions(board)

[['x' ' ' ' ']
 ['x' ' ' ' ']
 ['x' ' ' ' ']]


[1, 2, 4, 5, 7, 8]

In [289]:
def result(state, player, action):
    """Add move to the board."""
    
    state = state.copy()
    state[action] = player
  
    return state

show_board(empty_board())

print()
print("State for placing an x at position 4:")
show_board(result(empty_board(), 'x', 4))

[[' ' ' ' ' ']
 [' ' ' ' ' ']
 [' ' ' ' ' ']]

State for placing an x at position 4:
[[' ' ' ' ' ']
 [' ' 'x' ' ']
 [' ' ' ' ' ']]


In [290]:
def other(player): 
    if player == 'x': return 'o'
    else: return 'x'

In [291]:
def utility(state, player = 'x'):
    """check is a state is terminal and return the utility if it is. None means not a terminal mode."""
    goal = check_win(state)        
    if goal == player: return +1 
    if goal == 'd': return 0  
    if goal == other(player): return -1  # loss is failure
    return None # continue

print(utility(['x'] * 9))
print(utility(['o'] * 9))
print(utility(empty_board()))

1
-1
None


# Monte Carlo Tree Search with Upper Confidence Bound

See AIMA page 163. 

* Use a random playout policy.

__Important note:__ we use here a random playout policy, which ends up creating just a randomized search that works fine for this toy problem. For real applications you need to extend the code with a good __playout policy__ (e.g., learned by self-play)

## Simulate a Playout

In [292]:
def playout(state, action, player = 'x'):
    """Perfrom a random playout starting with the given action on the fiven board 
    and return the utility of the finished game."""
    state = result(state, player, action)
    current_player = other(player)
    
    while(True):
        # reached terminal state?
        u = utility(state, player)
        if u is not None: 
            return u
        
        # we use a random playout policy
        a = np.random.choice(get_actions(state))
        state = result(state, current_player, a)
        #print(state)
        
        # switch between players
        current_player = other(current_player)


board = empty_board()
display([ playout(board, 0) for i in range(20) ])

[1, 1, 0, 1, -1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, -1, 0, 1, 0, 1]

## Upper Confidence Bound Applied to Trees

Tree node definition.

In [293]:
class UCT_Node:
    def __init__(self, state, parent):
        self.state = state.copy()     # state description (a board)
        
        self.u = 0                    # utility
        self.n = 0                    # count
        
        self.parent = parent          # the parent
        self.children = {}            # dictionary with child nodes; keys are actions.
    
    def __str__(self):
        return f"UCT Node for state {self.state} with {len(self.children.keys())} children: ({self.u}/{self.n})"
    
    def UCB1(self):
        if(self.n < 1): return +math.inf
        if(self.parent is None): return +math.inf # CHECK!!!
        # self.u/self.n + C * math.sqrt(math.log(self.parent.n)/self.n)
        return self.u/self.n + C * math.sqrt(math.log(self.parent.n)/self.n)
    
n = UCT_Node(empty_board(), None)
n.n = 10
n.u = 5
print(n)

print(n.UCB1())

UCT Node for state [' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' '] with 0 children: (5/10)
inf


In [294]:
DEBUG = 1


class UCT:
    def __init__(self, C = math.sqrt(2)):
        self.tree = None
        self.leafs = []
        self.C = C
        
    def search(self, board, N = 100, player = 'x'):
        """Upper Confidence bound applied to Trees. 
        Simulation budget is N playouts."""
        global DEBUG
     
        self.tree = UCT_Node(board, None)
        self.leafs = [ self.tree ]
    
        actions = get_actions(board)
        u = {a: 0 for a in actions}
        n = {a: 0 for a in actions}
        n_parent = 0
    
        for i in range(N):

            # Select leaf node with highest UCB1
            # UCB1s = [ i.UCB1 for i in self.leafs ] 
            UCB1s = [i.UCB1() for i in self.leafs]
            node = self.leafs[UCB1s.index(max(UCB1s))]

            # node = self.leafs.index(max(UCB1s))
    
            # Expand: add a successor
            action_id = np.random.choice(range(len(actions)))
            action = actions[action_id]

            
            # TODO
        
            # Simulate
            # p = playout(board, actions[action_id], player = player)
            p = playout(board, action, player=player)

            # Back-Propagate (i.e., update counts and UCB1)
            # u[action_id] += p
            # n[action_id] += 1
            u[action] += p
            n[action] += 1

            n_parent += 1
        
            # for action_id in range(len(actions)):
            #     if n[action_id] > 0:
            #         UCB1[action_id] = u[action_id]/n[action_id] + self.C * math.sqrt(math.log(n_parent)/n[action_id])

            # Tính giá trị UCB1 cho từng hành động
            UCB1_values = []
            # for action_id in range(len(actions)):
            for action in actions:  
                # if n[action_id] > 0:
                #     value = u[action_id] / n[action_id] + self.C * math.sqrt(math.log(n_parent + 1) / n[action_id])
                # else:
                #     value = float('inf')
                # UCB1_values.append(value)

                if n[action] > 0:
                    value = u[action] / n[action] + self.C * math.sqrt(math.log(n_parent + 1) / n[action])
                else:
                    value = float('inf')
                UCB1_values.append(value)


    
        # if DEBUG >= 1: 
        #     print(pd.DataFrame({'action':actions, 
        #                         'total utility':u, 
        #                         '# of playouts':n, 
        #                         'UCB1':UCB1}))

        if DEBUG >= 1: 
            print(pd.DataFrame({
                'action': list(u.keys()),
                'total utility': list(u.values()),
                '# of playouts': list(n.values()),
                'UCB1': UCB1_values
            }))


    
        # return action with largest number of playouts
        # action = actions[n.index(max(n))]
        action = max(n, key=n.get)

        return action

board = empty_board()
display(board)

uct = UCT()

%timeit -n 1 -r 1 display(uct.search(board, N = 100))

[' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ']

   action  total utility  # of playouts      UCB1
0       0              6             13  1.304164
1       1             -1             12  0.793700
2       2              0             10  0.960741
3       3             -1             12  0.793700
4       4              4              6  1.906979
5       5              3             13  1.073395
6       6              5              8  1.699142
7       7              5              9  1.568266
8       8              2             17  0.854502


8

71.7 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


## Some Tests

In [295]:
def UCT_depth1(board, N=100):
    """Chạy UCT độ sâu 1 (thử từng nước đi N lần)."""
    actions = legal_moves(board)
    results = {}
    for a in actions:
        total = 0
        for _ in range(N):
            total += playout(board, a, player='x')  # hoặc tùy theo người chơi hiện tại
        results[a] = total / N
    return results


In [296]:
# x is about to win (play 8)

board = empty_board() 
board[0] = 'x'
board[1] = 'o'
board[3] = 'o'
board[4] = 'x'

print("Board:")
show_board(board)

print()
display(UCT_depth1(board ))

# print()
# uct = UCT()
# display(uct.search(board, N=100))

Board:
[['x' 'o' ' ']
 ['o' 'x' ' ']
 [' ' ' ' ' ']]



{2: 0.84, 5: 0.67, 6: 0.81, 7: 0.71, 8: 1.0}

In [297]:
# o is about to win

board = empty_board() 
board[0] = 'o'
board[1] = 'o'
board[3] = 'o'
board[4] = 'x'
board[8] = 'x'

print("Board:")
show_board(board)

print()
display(UCT_depth1(board))

Board:
[['o' 'o' ' ']
 ['o' 'x' ' ']
 [' ' ' ' 'x']]



{2: -0.08, 5: -0.6, 6: 0.06, 7: -0.78}

In [298]:
#### x can draw if it chooses 7.

board = empty_board() 
board[0] = 'x'
board[1] = 'o'
board[2] = 'x'
board[4] = 'o'

print("Board:")
show_board(board)

print()
display(UCT_depth1(board))

Board:
[['x' 'o' 'x']
 [' ' 'o' ' ']
 [' ' ' ' ' ']]



{3: -0.09, 5: -0.16, 6: -0.49, 7: 0.13, 8: -0.43}

In [299]:
# o went first

board = empty_board() 
board[4] = 'o'

print("Board:")
show_board(board)


print()
display(UCT_depth1(board))

Board:
[[' ' ' ' ' ']
 [' ' 'o' ' ']
 [' ' ' ' ' ']]



{0: -0.27,
 1: -0.55,
 2: -0.48,
 3: -0.62,
 5: -0.63,
 6: -0.51,
 7: -0.76,
 8: -0.42}

In [300]:
# Empty board: Only a draw an be guaranteed

board = empty_board() 

print("Board:")
show_board(board)


print()
%timeit -n 1 -r 1 display(UCT_depth1(board, N = 1000))

Board:
[[' ' ' ' ' ']
 [' ' ' ' ' ']
 [' ' ' ' ' ']]



{0: 0.391,
 1: 0.221,
 2: 0.327,
 3: 0.162,
 4: 0.519,
 5: 0.248,
 6: 0.304,
 7: 0.21,
 8: 0.342}

3.4 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [301]:
# A bad situation

board = empty_board() 
board[0] = 'o'
board[2] = 'x'
board[8] = 'o'

print("Board:")
show_board(board)


print()
display(UCT_depth1(board))

Board:
[['o' ' ' 'x']
 [' ' ' ' ' ']
 [' ' ' ' 'o']]



{1: -0.72, 3: -0.56, 4: 0.29, 5: -0.76, 6: -0.3, 7: -0.61}

__Note:__ It looks like random player o is very unlikely to block x and take advantage of the trap by playing the bottom left corner!

## Experiments


### Baseline: Randomized Player

A completely randomized player agent should be a weak baseline.

In [302]:
def random_player(board, player = None):
    """Simple player that chooses a random empy square. player is unused"""
    return np.random.choice(get_actions(board))

show_board(board)
random_player(board)

[['o' ' ' 'x']
 [' ' ' ' ' ']
 [' ' ' ' 'o']]


np.int64(4)

### The Environment

Implement the environment that calls the agent. The percept is the board and the action is move.

In [ ]:
DEBUG = 1

def switch_player(player, x, o):
    if player == 'x':
        return 'o', o
    else:
        return 'x', x

def play(x, o, N = 100):
    results = {'x': 0, 'o': 0, 'd': 0}
    for i in range(N):
        board = empty_board()
        player, fun = 'x', x
        
        while True:
            a = fun(board, player)
            board = result(board, player, a)
            
            win = check_win(board)
            if win != 'n':
                if DEBUG >= 1: print(f"{board} winner: {win}")
                results[win] += 1
                break
            
            player, fun = switch_player(player, x, o)   
 
    return results

### Random vs. Random

In [308]:
DEBUG = 0

%timeit -n 1 -r 1 display(play(random_player, random_player))

{'x': 57, 'o': 30, 'd': 13}

40.4 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


### Pure Monte Carlo Search vs. Random

In [311]:
# def uct10_player(board, player = 'x'):
#     action = UCT_depth1(board, N = 10, player = player)
#     return action

# def uct100_player(board, player = 'x'):
#     action = UCT_depth1(board, N = 100, player = player)
#     return action
def uct10_player(board, player='x'):
    results = UCT_depth1(board, N=10)
    action = max(results, key=results.get)  # lấy hành động có giá trị cao nhất
    return action

def uct100_player(board, player='x'):
    results = UCT_depth1(board, N=100)
    action = max(results, key=results.get)
    return action



DEBUG = 1
print("UCT vs. random:")
display(play(uct10_player, random_player, N = 1))

UCT vs. random:
['x', ' ', ' ', 'o', 'x', ' ', 'o', ' ', 'x'] winner: x


{'x': 1, 'o': 0, 'd': 0}

In [312]:
DEBUG = 0
print("UCT vs. random:")
%timeit -n 1 -r 1 display(play(uct10_player, random_player))

print()
print("random vs. UCT")
%timeit -n 1 -r 1 display(play(random_player, uct10_player))

UCT vs. random:


{'x': 95, 'o': 0, 'd': 5}

6.03 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)

random vs. UCT


{'x': 8, 'o': 67, 'd': 25}

4.12 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [313]:
DEBUG = 0
print("UCT vs. random:")
%timeit -n 1 -r 1 display(play(uct100_player, random_player))

print()
print("random vs. UCT")
%timeit -n 1 -r 1 display(play(random_player, uct100_player))

UCT vs. random:


{'x': 100, 'o': 0, 'd': 0}

1min ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)

random vs. UCT


{'x': 7, 'o': 64, 'd': 29}

41.8 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [314]:
DEBUG = 0
print("UCT (100) vs. UCT (10):")
%timeit -n 1 -r 1 display(play(uct100_player, uct10_player))

print()
print("UCT (10) vs. UCT (100)")
%timeit -n 1 -r 1 display(play(uct10_player, uct100_player))

UCT (100) vs. UCT (10):


{'x': 60, 'o': 0, 'd': 40}

1min 10s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)

UCT (10) vs. UCT (100)


{'x': 26, 'o': 9, 'd': 65}

48.8 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
